In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

In [2]:
using_colab = False

In [3]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib scikit-learn
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

In [4]:
%matplotlib widget

In [ ]:
import torch
import geopandas as gpd
from shapely.geometry import Polygon
from skimage import measure
import rasterio
from shapely.geometry import Polygon
import os
import datetime
import PIL.Image
import cv2
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
torch.inference_mode().__enter__()

# Load the model

In [6]:
import sam3
from sam3 import build_sam3_image_model
import os
sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")
bpe_path = f"{sam3_root}/sam3/assets/bpe_simple_vocab_16e6.txt.gz"

In [7]:
model = build_sam3_image_model(bpe_path=bpe_path)

In [8]:
from sam3.model.sam3_image_processor import Sam3Processor
processor = Sam3Processor(model)

# Jupyter widget

In [ ]:
import io

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import requests
from IPython.display import clear_output, display
from matplotlib.patches import Rectangle
from ipyfilechooser import FileChooser
from shapely.geometry import Point
import pandas as pd
from shapely.geometry import Polygon
from skimage import measure
import geopandas as gpd
import PIL.Image
import numpy as np
import os


class Sam3SegmentationWidget:
    """Interactive Jupyter widget for SAM3 segmentation with text and box prompts."""

    def __init__(self, processor):
        self.processor = processor
        self.state = None
        self.current_image = None
        self.current_image_array = None
        self.box_mode = "positive"
        self.drawing_box = False
        self.box_start = None
        self.current_rect = None
        self.image_counter = 0
        self.export_folder = None
        self.mask_registry = []
        self.active_mask_id = None

        self._setup_ui()
        self._setup_plot()

    # ------------------------------------------------------------------
    # UI SETUP
    # ------------------------------------------------------------------

    def _setup_ui(self):
        """Set up the UI components."""

        # ---- Context menu (ipywidgets-native, no JS) ------------------
        # This VBox is shown/hidden in place; it lives in a fixed-size
        # container next to the canvas so it always renders reliably.

        self._ctx_title = widgets.HTML(value="")

        self._ctx_species_dropdown = widgets.Dropdown(
            description="Species:",
            options=[],
            layout=widgets.Layout(width="220px"),
        )

        self._ctx_save_btn = widgets.Button(
            description="Save",
            button_style="success",
            icon="check",
            layout=widgets.Layout(width="90px"),
        )
        self._ctx_close_btn = widgets.Button(
            description="Close",
            icon="times",
            layout=widgets.Layout(width="90px"),
        )
        self._ctx_save_btn.on_click(self._ctx_on_save)
        self._ctx_close_btn.on_click(self._ctx_on_close)

        self._ctx_menu = widgets.VBox(
            [
                self._ctx_title,
                self._ctx_species_dropdown,
                widgets.HBox([self._ctx_save_btn, self._ctx_close_btn]),
            ],
            layout=widgets.Layout(
                display="none",           # hidden until right-click
                border="2px solid #555",
                border_radius="6px",
                padding="8px",
                background_color="white",
                width="250px",
                margin="4px 0 0 0",
            ),
        )

        # ---- Species panel -------------------------------------------
        self.species_input = widgets.Text(
            placeholder="Enter species name",
            description="Species:",
        )
        self.add_species_button = widgets.Button(
            description="Add species",
            button_style="success",
            icon="plus",
        )
        self.species_table_output = widgets.Output()
        self.species_list = []
        self.add_species_button.on_click(self._add_species)

        # ---- Image source controls -----------------------------------
        self.upload_widget = widgets.FileUpload(
            accept="image/*", multiple=False, description="Upload Image"
        )
        self.upload_widget.observe(self._on_image_upload, names="value")

        self.url_input = widgets.Text(placeholder="Or enter image URL")
        self.url_button = widgets.Button(description="Load URL", button_style="info")
        self.url_button.on_click(self._on_load_url)
        url_box = widgets.HBox(
            [self.url_input, self.url_button],
            layout=widgets.Layout(width="100%"),
        )

        # ---- Prompt controls -----------------------------------------
        self.text_input = widgets.Text(
            placeholder='Enter segmentation prompt (e.g., "trees", "dog")',
            continuous_update=False,
        )
        self.text_input.observe(self._on_text_submit, names="value")
        self.text_button = widgets.Button(description="Segment", button_style="success")
        self.text_button.on_click(self._on_text_prompt)
        text_box = widgets.HBox(
            [self.text_input, self.text_button],
            layout=widgets.Layout(width="100%"),
        )

        self.box_mode_buttons = widgets.ToggleButtons(
            options=["Positive Boxes", "Negative Boxes"],
            description="Box Mode:",
            tooltips=[
                "Draw boxes around objects to include",
                "Draw boxes around objects to exclude",
            ],
        )
        self.box_mode_buttons.observe(self._on_box_mode_change, names="value")

        self.clear_button = widgets.Button(
            description="Clear All Prompts", button_style="warning"
        )
        self.clear_button.on_click(self._on_clear_prompts)

        self.confidence_slider = widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01,
            description="Confidence:", continuous_update=False,
            style={"description_width": "initial"},
        )
        self.confidence_slider.observe(self._on_confidence_change, names="value")

        self.size_slider = widgets.IntSlider(
            value=960, min=300, max=2000, step=10,
            description="Image Size:", continuous_update=False,
            style={"description_width": "initial"},
        )
        self.size_slider.observe(self._on_size_change, names="value")

        self.folder_picker = FileChooser(
            os.getcwd(),
            title="Select export folder",
            show_only_dirs=True,
        )
        self.folder_picker.use_dir_icons = True
        self.folder_picker.default_filename = ""

        self.export_button = widgets.Button(
            description="Export Masks (GPKG)",
            button_style="primary",
            icon="download",
        )
        self.export_button.on_click(self._export_masks)

        # ---- Status / output -----------------------------------------
        self.output = widgets.Output()
        self.status_label = widgets.Label(value="Upload an image to begin")

        # CSS crosshair cursor
        css_style = widgets.HTML("""
        <style>
            .jupyter-matplotlib-canvas, canvas { cursor: crosshair !important; }
        </style>""")

        # ---- Accordion panes -----------------------------------------
        source_pane = widgets.VBox([self.upload_widget, url_box])
        prompt_pane = widgets.VBox([
            widgets.Label("Text Prompt:"),
            text_box,
            self.box_mode_buttons,
            self.confidence_slider,
            self.clear_button,
            self.folder_picker,
            self.export_button,
        ])
        display_pane = widgets.VBox([self.size_slider])

        self.accordion = widgets.Accordion(
            children=[source_pane, prompt_pane, display_pane]
        )
        self.accordion.set_title(0, "Image Source")
        self.accordion.set_title(1, "Segmentation Prompts")
        self.accordion.set_title(2, "Display Settings")
        self.accordion.selected_index = 0

        sidebar = widgets.VBox(
            [self.status_label, widgets.HTML("<h4>Controls</h4>"), self.accordion],
            layout=widgets.Layout(
                width="380px", min_width="380px", max_width="380px",
                border="1px solid #e0e0e0", padding="10px",
                margin="0 15px 0 0", flex="0 0 auto",
            ),
        )

        # The canvas output goes here; context menu sits just below it
        self.plot_container = widgets.Box([self.output])
        self.plot_container.add_class("no-drag")

        main_area = widgets.VBox(
            [self.plot_container],
            layout=widgets.Layout(flex="1", min_width="500px", overflow="auto"),
        )

        # Species panel (right column) – context menu lives at the bottom
        species_panel = widgets.VBox(
            [
                widgets.HTML("<b>Species Classes</b>"),
                self.species_input,
                self.add_species_button,
                self.species_table_output,
                widgets.HTML("<hr><b>Right-click menu</b>"),
                self._ctx_menu,          # <-- context menu lives here
            ],
            layout=widgets.Layout(
                width="280px", min_width="280px",
                border="1px solid #ddd", padding="10px",
                overflow="auto",
            ),
        )

        app_layout = widgets.HBox(
            [sidebar, main_area, species_panel],
            layout=widgets.Layout(
                width="100%", display="flex",
                flex_flow="row", align_items="stretch",
            ),
        )

        self.container = widgets.VBox([
            css_style,
            widgets.HTML("<h3>&#128444;&#65039; SAM3 Interactive Segmentation</h3>"),
            app_layout,
        ])

    # ------------------------------------------------------------------
    # CONTEXT MENU HELPERS (pure ipywidgets — no JS required)
    # ------------------------------------------------------------------

    def _show_context_menu(self, mask):
        """Populate and show the context menu for *mask*."""
        sid = mask["species_id"]
        if sid is None:
            label = "Unlabeled"
        else:
            label = next(
                (s["Species"] for s in self.species_list if s["Class ID"] == sid),
                "Unknown",
            )

        self._ctx_title.value = (
            f"<b>Mask {mask['mask_id']}</b> &nbsp;&mdash;&nbsp; "
            f"<span style='color:#555'>current: {label}</span>"
        )

        options = [(s["Species"], s["Class ID"]) for s in self.species_list]
        self._ctx_species_dropdown.options = options if options else [("(no species added)", None)]
        # pre-select current assignment
        if sid is not None and options:
            self._ctx_species_dropdown.value = sid

        self.active_mask_id = mask["mask_id"]
        self._ctx_menu.layout.display = "flex"   # make it visible

        # Scroll status bar so the user notices the panel
        self.status_label.value = (
            f"Right-click menu open for Mask {mask['mask_id']} "
            "→ see Species panel on the right"
        )

    def _ctx_on_save(self, _btn):
        """Save the selected species to the active mask."""
        if self.active_mask_id is None:
            return
        chosen = self._ctx_species_dropdown.value
        for m in self.mask_registry:
            if m["mask_id"] == self.active_mask_id:
                m["species_id"] = chosen
                break
        self._ctx_menu.layout.display = "none"
        self.active_mask_id = None
        # Refresh display so species labels repaint
        self._update_display()
        name = next(
            (s["Species"] for s in self.species_list if s["Class ID"] == chosen),
            str(chosen),
        )
        self.status_label.value = f"Saved: mask labelled as '{name}'"

    def _ctx_on_close(self, _btn):
        """Close the context menu without saving."""
        self._ctx_menu.layout.display = "none"
        self.active_mask_id = None
        self.status_label.value = "Context menu closed"

    # ------------------------------------------------------------------
    # PLOT SETUP
    # ------------------------------------------------------------------

    def _setup_plot(self):
        self.fig, self.ax = plt.subplots(figsize=(12, 8))
        self.ax.axis("off")
        self.fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        self.fig.canvas.toolbar_visible = False
        self.fig.canvas.header_visible = False
        self.fig.canvas.footer_visible = False
        self.fig.canvas.resizable = False

    # ------------------------------------------------------------------
    # LOADING STATE
    # ------------------------------------------------------------------

    def _set_loading(self, is_loading, message="Processing..."):
        if is_loading:
            self.status_label.value = f"⏳ {message}"
            for w in (self.upload_widget, self.url_button, self.text_button,
                      self.clear_button, self.box_mode_buttons, self.confidence_slider):
                w.disabled = True
        else:
            for w in (self.upload_widget, self.url_button, self.text_button,
                      self.clear_button, self.box_mode_buttons, self.confidence_slider):
                w.disabled = False

    # ------------------------------------------------------------------
    # IMAGE LOADING
    # ------------------------------------------------------------------

    def _on_image_upload(self, change):
        if change["new"]:
            uploaded_file = change["new"][0]
            self.image_name = os.path.splitext(uploaded_file["name"])[0]
            image = PIL.Image.open(io.BytesIO(uploaded_file["content"])).convert("RGB")
            self._set_image(image)

    def _on_load_url(self, button):
        url = self.url_input.value.strip()
        if not url:
            self.status_label.value = "Please enter a URL"
            return
        self._set_loading(True, "Downloading image from URL...")
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            image = PIL.Image.open(io.BytesIO(response.content)).convert("RGB")
            self._set_image(image)
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error loading image: {str(e)}"

    def _set_image(self, image):
        self._set_loading(True, "Processing image through model...")
        try:
            self.current_image = image
            self.current_image_array = np.array(image)
            self.state = self.processor.set_image(image)
            self._set_loading(False)
            self.status_label.value = f"Image loaded: {image.size[0]}x{image.size[1]} pixels"
            self._resize_figure()
            self._update_display()
            self._connect_plot_events()
            self.accordion.selected_index = 1
            self.image_counter += 1
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error processing image: {str(e)}"

    # ------------------------------------------------------------------
    # SPECIES MANAGEMENT
    # ------------------------------------------------------------------

    def _add_species(self, button):
        name = self.species_input.value.strip()
        if not name:
            return
        class_id = len(self.species_list) + 1
        self.species_list.append({"Class ID": class_id, "Species": name})
        df = pd.DataFrame(self.species_list)
        with self.species_table_output:
            self.species_table_output.clear_output()
            display(df)
        self.species_input.value = ""
        # Refresh dropdown options in the context menu
        self._ctx_species_dropdown.options = [
            (s["Species"], s["Class ID"]) for s in self.species_list
        ]

    # ------------------------------------------------------------------
    # PROMPT HANDLERS
    # ------------------------------------------------------------------

    def _on_text_submit(self, change):
        self._on_text_prompt(None)

    def _on_text_prompt(self, button):
        if self.state is None:
            self.status_label.value = "Please load an image first"
            return
        prompt = self.text_input.value.strip()
        if not prompt:
            self.status_label.value = "Please enter a prompt"
            return
        self._set_loading(True, f'Segmenting with prompt: "{prompt}"...')
        try:
            self.state = self.processor.set_text_prompt(prompt, self.state)
            self._set_loading(False)
            self.status_label.value = f'Segmented with prompt: "{prompt}"'
            self._update_display()
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error: {str(e)}"

    def _on_box_mode_change(self, change):
        self.box_mode = "positive" if change["new"] == "Positive Boxes" else "negative"

    def _on_clear_prompts(self, button):
        if self.current_image is not None:
            try:
                self._set_loading(True, "Clearing prompts and resetting...")
                self.state = self.processor.reset_all_prompts(self.state)
                if "prompted_boxes" in self.state:
                    del self.state["prompted_boxes"]
                self.text_input.value = ""
                self._set_loading(False)
                self.status_label.value = "Cleared all prompts"
                self._update_display()
            except Exception as e:
                self._set_loading(False)
                import traceback
                self.status_label.value = f"Error: {str(e)} {traceback.format_exc()}"

    def _on_confidence_change(self, change):
        if self.state is not None:
            self.state = self.processor.set_confidence_threshold(change["new"], self.state)
            self._update_display()

    # ------------------------------------------------------------------
    # MATPLOTLIB EVENT HANDLERS
    # ------------------------------------------------------------------

    def _connect_plot_events(self):
        if hasattr(self.fig.canvas, "toolbar") and self.fig.canvas.toolbar is not None:
            self.fig.canvas.toolbar.pan()
            self.fig.canvas.toolbar.pan()
        self.fig.canvas.mpl_connect("button_press_event",   self._on_press)
        self.fig.canvas.mpl_connect("button_release_event", self._on_release)
        self.fig.canvas.mpl_connect("motion_notify_event",  self._on_motion)
        self.fig.canvas.mpl_connect("motion_notify_event",  self._on_mask_hover)

    def _on_press(self, event):
        if event.inaxes != self.ax:
            return
        # Right-click (button 3) or middle-click (button 2) → context menu
        if event.button in (2, 3):
            self._on_mask_click(event)
            return
        # Left-click → start drawing box
        self.drawing_box = True
        self.box_start = (event.xdata, event.ydata)

    def _on_motion(self, event):
        if not self.drawing_box or event.inaxes != self.ax or self.box_start is None:
            return
        if self.current_rect is not None:
            self.current_rect.remove()
        x0, y0 = self.box_start
        x1, y1 = event.xdata, event.ydata
        color = "green" if self.box_mode == "positive" else "red"
        self.current_rect = Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            fill=False, edgecolor=color, linewidth=2, linestyle="--",
        )
        self.ax.add_patch(self.current_rect)
        self.fig.canvas.draw_idle()

    def _on_release(self, event):
        if not self.drawing_box or event.inaxes != self.ax or self.box_start is None:
            self.drawing_box = False
            return
        self.drawing_box = False
        if self.current_rect is not None:
            self.current_rect.remove()
            self.current_rect = None
        if self.state is None:
            return
        x0, y0 = self.box_start
        x1, y1 = event.xdata, event.ydata
        x_min, x_max = min(x0, x1), max(x0, x1)
        y_min, y_max = min(y0, y1), max(y0, y1)
        if abs(x_max - x_min) < 5 or abs(y_max - y_min) < 5:
            return
        img_h = self.state["original_height"]
        img_w = self.state["original_width"]
        center_x = (x_min + x_max) / 2.0 / img_w
        center_y = (y_min + y_max) / 2.0 / img_h
        width    = (x_max - x_min) / img_w
        height   = (y_max - y_min) / img_h
        box   = [center_x, center_y, width, height]
        label = self.box_mode == "positive"
        mode_str = "positive" if label else "negative"
        if "prompted_boxes" not in self.state:
            self.state["prompted_boxes"] = []
        self.state["prompted_boxes"].append(
            {"box": [x_min, y_min, x_max, y_max], "label": label}
        )
        self._set_loading(True, f"Adding {mode_str} box and re-segmenting...")
        try:
            self.state = self.processor.add_geometric_prompt(box, label, self.state)
            self._set_loading(False)
            self.status_label.value = f"Added {mode_str} box"
            self._update_display()
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error adding box: {str(e)}"

    def _on_mask_click(self, event):
        """Open the context menu when the user right-clicks on a mask."""
        if event.xdata is None or event.ydata is None:
            return
        point = Point(event.xdata, event.ydata)
        for mask in self.mask_registry:
            if mask["polygon"].contains(point):
                self._show_context_menu(mask)
                return
        # Click was outside all masks — close any open menu
        self._ctx_menu.layout.display = "none"

    def _on_mask_hover(self, event):
        if event.xdata is None or event.ydata is None:
            return
        point = Point(event.xdata, event.ydata)
        for mask in self.mask_registry:
            if mask["polygon"].buffer(2).contains(point):
                sid = mask["species_id"]
                if sid is None:
                    self.status_label.value = f"Mask {mask['mask_id']} (unlabeled) — right-click to label"
                else:
                    name = next(
                        (s["Species"] for s in self.species_list if s["Class ID"] == sid),
                        "?",
                    )
                    self.status_label.value = f"Mask {mask['mask_id']} → {name}"
                return

    # ------------------------------------------------------------------
    # DISPLAY
    # ------------------------------------------------------------------

    def _resize_figure(self):
        if self.current_image is None:
            return
        img_w, img_h = self.current_image.size
        display_w = float(self.size_slider.value)
        aspect_ratio = img_h / img_w
        display_h = int(display_w * aspect_ratio)
        dpi = self.fig.dpi
        self.fig.set_size_inches(display_w / dpi, display_h / dpi, forward=True)

    def _on_size_change(self, change):
        if self.current_image is not None:
            self._resize_figure()
            self._update_display()

    def _update_display(self):
        if self.current_image_array is None:
            return

        with self.output:
            clear_output(wait=True)
            self.ax.clear()
            self.ax.axis("off")
            self.ax.imshow(self.current_image_array)

            if self.state is not None and "masks" in self.state:
                masks  = self.state.get("masks", [])
                boxes  = self.state.get("boxes", [])
                scores = self.state.get("scores", [])

                if len(masks) > 0:
                    # Preserve existing species labels across redraw
                    prev_labels = {
                        m["mask_id"]: m["species_id"]
                        for m in self.mask_registry
                    }
                    self.mask_registry = []

                    mask_overlay = np.zeros(
                        (*self.current_image_array.shape[:2], 4)
                    )

                    for i, (mask_t, box, score) in enumerate(zip(masks, boxes, scores)):
                        mask_np = mask_t[0].cpu().numpy()
                        color   = plt.cm.tab10(i % 10)[:3]
                        mask_overlay[mask_np > 0.5] = (*color, 0.5)

                        x0, y0, x1, y1 = box.cpu().numpy()
                        self.ax.add_patch(Rectangle(
                            (x0, y0), x1 - x0, y1 - y0,
                            fill=False, edgecolor=color, linewidth=2,
                        ))
                        self.ax.text(
                            x0, y0 - 5, f"{score:.2f}",
                            color="white", fontsize=10,
                            bbox=dict(facecolor=color, alpha=0.7,
                                      edgecolor="none", pad=2),
                        )

                        # Build polygon for hit-testing
                        binary   = mask_np > 0.5
                        contours = measure.find_contours(binary, 0.5)
                        for contour in contours:
                            coords = [(c[1], c[0]) for c in contour]
                            if len(coords) > 3:
                                poly = Polygon(coords)
                                if poly.is_valid:
                                    self.mask_registry.append({
                                        "mask_id":    i,
                                        "polygon":    poly,
                                        "species_id": prev_labels.get(i),
                                    })
                                    break  # one polygon per mask is enough

                    self.ax.imshow(mask_overlay)

                    # Draw species name labels on the canvas
                    for m in self.mask_registry:
                        if m["species_id"] is not None:
                            cx = m["polygon"].centroid.x
                            cy = m["polygon"].centroid.y
                            name = next(
                                (s["Species"] for s in self.species_list
                                 if s["Class ID"] == m["species_id"]),
                                "?",
                            )
                            self.ax.text(
                                cx, cy, name,
                                color="yellow", fontsize=10, ha="center",
                                bbox=dict(facecolor="black", alpha=0.6),
                            )

                    self.status_label.value = (
                        f"Found {len(masks)} object(s) "
                        "— right-click a mask to assign a species"
                    )
                else:
                    self.status_label.value = "No objects found above confidence threshold"

            # Prompted boxes (dashed outlines)
            if self.state is not None and "prompted_boxes" in self.state:
                for pb in self.state["prompted_boxes"]:
                    x0, y0, x1, y1 = pb["box"]
                    color = "green" if pb["label"] else "red"
                    self.ax.add_patch(Rectangle(
                        (x0, y0), x1 - x0, y1 - y0,
                        fill=False, edgecolor=color, linewidth=2, linestyle="--",
                    ))

            display(self.fig.canvas)

    # ------------------------------------------------------------------
    # EXPORT
    # ------------------------------------------------------------------

    def _ensure_export_folder(self):
        selected = self.folder_picker.selected_path
        if not selected:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            folder = os.path.join(os.getcwd(), f"sam3_exports_{timestamp}")
            os.makedirs(folder, exist_ok=True)
        else:
            folder = selected
        self.export_folder = folder
        print(f"\nMasks will be exported to:\n{self.export_folder}\n")

    def _export_masks(self, button):
        if self.state is None or "masks" not in self.state:
            self.status_label.value = "No masks available"
            return
        masks = self.state["masks"]
        if len(masks) == 0:
            self.status_label.value = "No masks detected"
            return

        self._ensure_export_folder()

        img_h = self.state["original_height"]
        img_w = self.state["original_width"]
        combined_mask = np.zeros((img_h, img_w), dtype=np.uint8)
        polygons, ids = [], []

        for i, mask in enumerate(masks):
            mask_np = mask[0].cpu().numpy()
            binary  = mask_np > 0.5
            combined_mask[binary] = i + 1
            mask_uint8 = (binary.astype(np.uint8) * 255)
            contours, _ = cv2.findContours(
                mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            for contour in contours:
                coords = [(int(pt[0][0]), img_h - int(pt[0][1])) for pt in contour]
                if len(coords) > 3:
                    poly = Polygon(coords)
                    if poly.is_valid:
                        polygons.append(poly)
                        ids.append(i)

        base_name = getattr(self, "image_name", "sam3_image")

        png_path = os.path.join(self.export_folder, f"{base_name}_sam3_masks.png")
        png_mask = ((combined_mask.astype(np.float32) / max(combined_mask.max(), 1)) * 255).astype(np.uint8)
        PIL.Image.fromarray(png_mask).save(png_path)

        if polygons:
            records = [
                {
                    "mask_id":  m["mask_id"],
                    "class_id": m["species_id"],
                    "species":  next(
                        (s["Species"] for s in self.species_list if s["Class ID"] == m["species_id"]),
                        "Unlabeled",
                    ),
                    "geometry": m["polygon"],
                }
                for m in self.mask_registry
                if m["species_id"] is not None
            ]
            if records:
                gpkg_path = os.path.join(self.export_folder, f"{base_name}_sam3_masks.gpkg")
                gpd.GeoDataFrame(records, geometry="geometry").to_file(gpkg_path, driver="GPKG")
                print(f"GPKG exported: {gpkg_path}")
            else:
                self.status_label.value = "No masks have species assigned — GPKG not written"

        print(f"PNG exported: {png_path}")
        self.status_label.value = (
            f"Exported masks for image {self.image_counter} → "
            f"{base_name}_sam3_masks.png / .gpkg"
        )

    # ------------------------------------------------------------------
    # DISPLAY ENTRY POINTS
    # ------------------------------------------------------------------

    def display(self):
        display(self.container)

    def _ipython_display_(self):
        self.display()


# Run!

In [ ]:
widget = Sam3SegmentationWidget(processor)
widget.display()